# 04 · Transformer encoder (self-attention)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LekshmiSR/proteins-d1-hands-on/blob/main/notebooks/04_transformer.ipynb)

**Idea.** In a Transformer, every residue looks at **every other residue in the protein at once** and decides how much attention to pay to each of them. This is called **self-attention**. Unlike the LSTM, nothing is processed step by step, so it runs fully in parallel on a GPU, and unlike the CNN, distance does not limit what a residue can see.

```
residue i asks a question (query)    "who is relevant to me?"
every residue j offers a key          "this is what I am"
         score(i, j) = query_i · key_j          → softmax over j → attention weights (sum to 1)
every residue j offers a value        "this is my information"
         new_i = Σ_j weight(i, j) × value_j
```

Transformers are the architecture behind large language models and protein language models such as ESM and ProtT5.

This notebook builds attention **by hand** first, then uses PyTorch's ready-made Transformer layers.

Expected time on a Colab T4: about 6–10 minutes.

## 0. Setup

Run the next cell first. It works in two places:

* **Google Colab**: it downloads this repository (code + helpers) into the Colab machine.
* **Your own computer** (Jupyter / VS Code): it just moves to the repository folder.

> **Colab tip:** use a GPU. Menu **Runtime → Change runtime type → T4 GPU → Save**. Do this *before* running cells.

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/LekshmiSR/proteins-d1-hands-on.git"
REPO_DIR = REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")

if "google.colab" in sys.modules:                       # running in Colab
    if os.path.basename(os.getcwd()) != REPO_DIR:
        if not os.path.exists(REPO_DIR):
            subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
        os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":      # running locally from notebooks/
    os.chdir("..")

sys.path.insert(0, os.getcwd())
print("Working folder:", os.getcwd())

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from protein_dl import data, train, plots

train.set_seed(42)
device = train.get_device()
import math

## 1. Settings

In [ ]:
D_MODEL = 128        # size of the vector for each residue throughout the network
N_HEADS = 4          # attention heads (D_MODEL must be divisible by N_HEADS)
FF_DIM = 256         # hidden size of the feed-forward block inside each layer
N_LAYERS = 3         # stacked Transformer layers
DROPOUT = 0.1

EPOCHS = 15
BATCH_SIZE = 32
LR = 5e-4            # Transformers like a slightly smaller learning rate
MAX_LEN = 512
TRAIN_SUBSET = None  # e.g. 2000 for a quick test
SAVE_TO_DRIVE = False

## 2. Load the data

The same loaders as in notebook 00 (see there for every detail). We grab one batch to push through the model step by step.

In [ ]:
train_dl, valid_dl, test_dl = data.make_dataloaders("data", batch_size=BATCH_SIZE, max_len=MAX_LEN,
                                                     train_subset=TRAIN_SUBSET)
tokens, labels, lengths = next(iter(train_dl))
print("tokens :", tokens.shape, "  <- [B, L]")
print("labels :", labels.shape)
print("lengths:", lengths.shape)

## 3. Build the model step by step

### Step 1: embedding

In [ ]:
embedding = nn.Embedding(data.VOCAB_SIZE, D_MODEL, padding_idx=data.PAD_IDX)
x = embedding(tokens) * math.sqrt(D_MODEL)    # the usual scaling from the original Transformer paper
print("embedded:", tuple(x.shape), "  [B, L, D]")

### Step 2: positional encoding

Self-attention by itself has **no notion of order**: if you shuffled the residues, each one would get the same result. But order is everything in a protein. So we **add** to each residue a vector that encodes its position. The classic choice uses sine and cosine waves of different frequencies: low dimensions change quickly along the chain, high dimensions change slowly.

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=4096):
        super().__init__()
        pos = torch.arange(max_len).unsqueeze(1)                                   # [max_len, 1]
        freq = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(pos * freq)                                        # even dimensions
        pe[:, 1::2] = torch.cos(pos * freq)                                        # odd dimensions
        self.register_buffer("pe", pe)                                             # saved, but not trained

    def forward(self, x):                                                          # x: [B, L, D]
        return x + self.pe[: x.shape[1]]


posenc = PositionalEncoding(D_MODEL)
x = posenc(x)
print("after adding positions:", tuple(x.shape))

plt.figure(figsize=(8, 3))
plt.imshow(posenc.pe[:200].T, aspect="auto", cmap="RdBu")
plt.xlabel("residue position"); plt.ylabel("dimension"); plt.colorbar()
plt.title("Positional encoding: each column is the vector added at that position")
plt.show()

### Step 3: self-attention by hand (one head)

Three linear layers turn every residue vector into a **query**, a **key** and a **value**. Then:

1. **scores** = Q · Kᵀ / √d: a `[L, L]` table per protein, "how relevant is residue j to residue i".
2. **mask** the padding: set the scores of padded positions to −∞ so they get zero attention.
3. **softmax** over j: each row becomes weights that sum to 1.
4. **output** = weights · V: each residue gets a weighted mix of all residues' values.

In [ ]:
d = D_MODEL
W_q, W_k, W_v = nn.Linear(d, d), nn.Linear(d, d), nn.Linear(d, d)

Q, K, V = W_q(x), W_k(x), W_v(x)
print("Q, K, V       :", tuple(Q.shape), "  [B, L, D]")

scores = Q @ K.transpose(1, 2) / math.sqrt(d)
print("scores        :", tuple(scores.shape), "  [B, L, L]  one row per residue, one column per residue")

pad_mask = tokens == data.PAD_IDX                     # [B, L]  True where there is padding
scores = scores.masked_fill(pad_mask[:, None, :], float("-inf"))

weights = scores.softmax(dim=-1)
print("weights       :", tuple(weights.shape), "  each row sums to", round(weights[0, 0].sum().item(), 4))

out = weights @ V
print("output        :", tuple(out.shape), "  [B, L, D]  same shape as the input")

i = int(lengths.argmin())
print(f"\nprotein {i} has {int(lengths[i])} residues; attention residue 0 gives to padding:",
      weights[i, 0, lengths[i]:].sum().item())

### Step 4: multi-head attention

One attention pattern is limited: a residue may want to look at its *i±4* neighbours (helix) and, separately, at a distant strand partner. **Multi-head attention** runs several smaller attentions in parallel (here 4 heads of size 32) and concatenates their outputs. PyTorch provides it as `nn.MultiheadAttention`:

In [ ]:
mha = nn.MultiheadAttention(D_MODEL, N_HEADS, batch_first=True)
out, attn = mha(x, x, x, key_padding_mask=pad_mask, need_weights=True, average_attn_weights=False)
print("output           :", tuple(out.shape), "  [B, L, D]")
print("attention weights:", tuple(attn.shape), "  [B, heads, L, L]")

### Step 5: a full Transformer layer

One encoder layer = **attention** + a small **feed-forward network** (applied to each residue separately), each wrapped with:
* a **residual connection** (`x + sublayer(x)`): the layer only has to learn a *change* to its input, which makes deep networks train well;
* **layer normalisation**: keeps the numbers in a stable range.

```
x → LayerNorm → Multi-head attention → + x → LayerNorm → Linear-ReLU-Linear → + → next layer
```

`nn.TransformerEncoderLayer` implements exactly this. `norm_first=True` puts the LayerNorm before each block ("pre-norm"), which trains more stably.

In [ ]:
layer = nn.TransformerEncoderLayer(D_MODEL, N_HEADS, dim_feedforward=FF_DIM, dropout=DROPOUT,
                                   batch_first=True, norm_first=True)
print(layer)
y = layer(x, src_key_padding_mask=pad_mask)
print("\nlayer output:", tuple(y.shape), "  [B, L, D]  (same shape, so layers can be stacked)")

### The model class

In [ ]:
class TransformerSS(nn.Module):
    def __init__(self, d_model=D_MODEL, n_heads=N_HEADS, ff_dim=FF_DIM, n_layers=N_LAYERS, dropout=DROPOUT):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(data.VOCAB_SIZE, d_model, padding_idx=data.PAD_IDX)
        self.posenc = PositionalEncoding(d_model)
        self.dropout = nn.Dropout(dropout)
        layer = nn.TransformerEncoderLayer(d_model, n_heads, dim_feedforward=ff_dim, dropout=dropout,
                                           batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, data.NUM_CLASSES)

    def forward(self, tokens, lengths=None):
        pad_mask = tokens == data.PAD_IDX                                      # [B, L]
        x = self.embedding(tokens) * math.sqrt(self.d_model)                   # [B, L, D]
        x = self.dropout(self.posenc(x))
        x = self.encoder(x, src_key_padding_mask=pad_mask)                     # [B, L, D]
        return self.head(self.norm(x))                                         # [B, L, 3]


model = TransformerSS().to(device)
print(f"Trainable parameters: {train.count_parameters(model):,}")
for name, p in model.named_parameters():
    if name.startswith("encoder.layers.0"):
        print(f"  {name:45s} {tuple(p.shape)}")
print("  ... and the same for layers 1 and 2")
with torch.no_grad():
    print("output shape on our batch:", tuple(model(tokens.to(device), lengths).shape))

## Train

`train.train_model` runs the loop from notebook 01: for each epoch it trains on all training batches, then measures loss and Q3 on the validation set, and remembers the weights of the best epoch. Transformers trained from scratch are often a bit unstable at the start; the learning-rate schedule and gradient clipping inside `train_model` help.

In [ ]:
train.set_seed(42)
history = train.train_model(model, train_dl, valid_dl, device, epochs=EPOCHS, lr=LR)

In [ ]:
plots.plot_history(history, "Transformer")

**Reading the curves:** training loss should go down steadily. If validation loss starts going *up* while training loss keeps going down, the model is **overfitting** (memorising the training proteins). The best-epoch weights protect us from that.

## Evaluate on the test set (CB513)

The model now has the weights from its best validation epoch. We run it **once** on CB513, the independent test set.

In [ ]:
val_metrics = train.evaluate(model, valid_dl, device)
test_metrics = train.evaluate(model, test_dl, device)

print(f"Validation Q3: {val_metrics['q3_accuracy']:.4f}")
print(f"CB513 Q3     : {test_metrics['q3_accuracy']:.4f}   macro F1: {test_metrics['macro_f1']:.4f}")
print()
print("Per-class scores on CB513:")
for cls, s in test_metrics["per_class"].items():
    print(f"  {cls:6s} precision {s['precision']:.3f}  recall {s['recall']:.3f}  F1 {s['f1']:.3f}  ({s['support']} residues)")

In [ ]:
plots.plot_confusion(test_metrics["confusion_matrix"], "Transformer on CB513 (rows sum to 1)")

**How to read the confusion matrix:** each row is a true class and shows where its residues ended up. The diagonal is the recall of each class. Off-diagonal cells show the typical mistakes (strands predicted as coil is usually the biggest one).

Let us look at a real prediction for one CB513 protein:

In [ ]:
trues, preds, _ = train.predict(model, test_dl, device)
i = 100                                           # try other numbers!
x = np.array(test_dl.dataset.tokens[i]); y = np.array(test_dl.dataset.labels[i])
seq = "".join(data.AMINO_ACIDS[t - 1] if 1 <= t <= 20 else "X" for t in x[y != data.IGNORE_INDEX])
print(f"protein {i}: {len(seq)} scored residues, accuracy {(trues[i] == preds[i]).mean():.2f}\n")
plots.show_prediction(seq, trues[i], preds[i])

## Save the results

The results (metrics, training curves, settings) are written to `results/transformer.json`. Notebook 05 reads these files to compare all models.

In Colab, files disappear when the session ends. Set `SAVE_TO_DRIVE = True` in the settings cell at the top to also keep a copy in your Google Drive (Colab will ask for permission).

In [ ]:
drive_dir = train.get_drive_dir(SAVE_TO_DRIVE)
config = dict(EPOCHS=EPOCHS, BATCH_SIZE=BATCH_SIZE, LR=LR, MAX_LEN=MAX_LEN, TRAIN_SUBSET=TRAIN_SUBSET)
results = train.save_results("transformer", model, history, val_metrics, test_metrics, config=config, drive_dir=drive_dir)

## Look inside: what does a trained residue attend to?

We take the first layer of the trained model and plot its attention weights (averaged over heads) for one validation protein. Bright cells mean "residue on the y-axis pays attention to residue on the x-axis".

In [ ]:
model.eval()
with torch.no_grad():
    vt, vl, vlen = next(iter(valid_dl))                                   # shortest validation proteins
    k = len(vlen) - 1                                                     # longest one in this batch
    t = vt[k:k+1, : vlen[k]].to(device)
    h = model.dropout(model.posenc(model.embedding(t) * math.sqrt(model.d_model)))
    first = model.encoder.layers[0]
    _, w = first.self_attn(first.norm1(h), first.norm1(h), first.norm1(h), need_weights=True)

plt.figure(figsize=(5, 4.5))
plt.imshow(w[0].cpu(), cmap="viridis")
plt.xlabel("attended residue (key)"); plt.ylabel("residue (query)"); plt.colorbar()
plt.title(f"Layer 1 attention, protein of {int(vlen[k])} residues")
plt.show()

Often you will see a bright **diagonal band**: residues attend mostly to their close neighbours, which is exactly the local context that helices and turns depend on. Other patterns differ from head to head and layer to layer.

## What did we learn?

* The Transformer can relate any two residues directly and trains in parallel.
* But it has **no built-in idea of locality** (the CNN gets it for free from its small filters) and must learn it from data. With only ~18k proteins and single sequences, a from-scratch Transformer often does **not** beat the CNN or the BiLSTM. Transformers shine when pretrained on millions of sequences: that is what protein language models such as **ESM-2** do. Using their embeddings as input lifts Q3 well above all models here.

## Exercises

1. Remove the positional encoding (`x = self.dropout(x)` instead of `self.posenc(x)`). What happens to Q3? Why?
2. Try `N_LAYERS = 1` and `N_LAYERS = 6`, or `N_HEADS = 1` and `8`.
3. Hybrid idea: add a `Conv1d` (kernel 9) before the encoder to give the model local context for free. Does it help?